Phase 3 – Window Functions (Senior SQL)

This is where most Data Engineering interviews spend a lot of time.

We'll progress from Easy → Medium → Hard, just like LeetCode.

========================================================================================


Module 3.1 – Ranking Functions

Topics:

ROW_NUMBER()
RANK()
DENSE_RANK()
PARTITION BY
ORDER BY inside windows

Questions:
Q036,
Q037,
Q038,
Q039,
Q040

===================================================================================



Q036 (Window Functions – Easy)
Problem

Using the employees table:

Display each employee's:

Employee Name,
Department Name,
Salary,
Salary Rank within the department

Highest salary should have Rank = 1.

Rules: 

* Use a window function.

* Do not use subqueries or CTEs.

* If two employees have the same salary, think about whether RANK() or DENSE_RANK() best matches the requirement, and be prepared to explain your choice.

Note: This question introduces one of the most frequently asked SQL interview topics, and we'll build the rest of the window function module from here.

In [0]:
%sql

EXPLAIN FORMATTED
select e.emp_name, d.dept_name, e.salary,
            dense_rank() over(partition by e.dept_id order by salary desc) as rank
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id

Q037 (Easy → Medium)
Problem

Display the following for every employee:

Employee Name,
Department Name,
Salary,
Department Highest Salary

Rules:

Do not use GROUP BY.


Do not use a subquery.


Do not use a CTE.


Solve it using a window function.

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
            max(e.salary) over(partition by e.dept_id) as department_highest_salary
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id

Q038 (Medium)

This is one of the most frequently asked window function questions.

Problem

Display:

Employee Name,
Department Name,
Salary,
Department Average Salary,
Salary Difference from Department Average

Rules:

* No GROUP BY

* No CTE

* No subquery

* Use window functions only

In [0]:
%sql

-- EXPLAIN EXTENDED
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        cast(avg(e.salary) over(partition by d.dept_id) as decimal(10,2)) as department_average_salary, (salary - department_average_salary) as department_salary_difference
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id


Q039 — Multiple Window Functions (Medium)
Problem

Using the employees and departments tables, display:

Employee Name

Department Name

Salary

Salary Rank within the Department

Highest Salary in the Department

Difference between the Employee's Salary and the Department Highest Salary

Rules:
✅ No GROUP BY
✅ No CTE
✅ No subquery
✅ Use only window functions

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
            dense_rank() over(partition by e.dept_id order by salary desc) as rank,
            max(e.salary) over(partition by e.dept_id) as department_highest_salary,
            (e.salary - max(e.salary) over(partition by e.dept_id)) as highest_department_salary_difference
        
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id


🚀 Q040 (Medium)

Using the employees table, display:

Employee Name

Department Name

Salary

Previous Employee's Salary within the same department

Salary Difference from the Previous Employee

Rules:

Use LAG().

Partition by department.

Order employees by salary descending.

Do not use joins, CTEs, or subqueries beyond the required join to get the department name.

This is one of the most frequently asked interview questions because it introduces row-to-row comparisons using window functions.

In [0]:
%sql

EXPLAIN FORMATTED
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
            lag(e.salary, 1, e.salary) over(partition by d.dept_id order by salary desc) as previous_employee_salary,
            (salary - previous_employee_salary) as salary_difference_from_previous_employee
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id


Find departments where there was a hiring gap of more than 365 days.

In [0]:
%sql

with hiring_gap as (
    select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date,
            lead(e.hire_date) over(partition by d.dept_id order by e.hire_date, e.emp_id) as next_date,
            date_diff(next_date, hire_date) as employee_hire_gap_in_department
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
)
select * from hiring_gap where employee_hire_gap_in_department > 365



Module 3.2 – Running Calculations

These questions teach you how to define window frames, which is one of the most misunderstood SQL topics.

Topics:

SUM() OVER()
AVG() OVER()
Running Total
Cumulative Average
Moving Average

Questions:
Q041,
Q042,
Q043,
Q044

===================================================================================



Q041 — Running Total (Medium)
Problem

Using the employees table, display:

Employee Name,
Department Name,
Hire Date,
Salary,
Running Total Salary within each department

The running total should be calculated according to hire date.

Rules: 
✅ No GROUP BY
✅ No CTE
✅ No subquery
✅ Use SUM() OVER()
✅ Order by hire_date
✅ Partition by department

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        sum(e.salary) over(partition by d.dept_id) as total_salary
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'
    



In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        sum(e.salary) over(partition by d.dept_id order by e.hire_date
                ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING) as total_salary
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'
    

🚀 Q042 (Senior-Level Window Functions)

Now let's make it more challenging.

Using the employees table, display:

Employee Name,
Department Name,
Hire Date,
Salary,
Moving Average Salary (Current Employee + Previous 2 Employees)

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        cast(avg(e.salary) over(partition by d.dept_id  order by e.hire_date, e.emp_id
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as decimal(10,2)) as moving_avg_salary_preceding_2
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        sum(e.salary) over(partition by d.dept_id  order by e.salary desc
                RANGE BETWEEN 10000 PRECEDING AND 5000 FOLLOWING) as moving_total_salary
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        count(e.emp_id) over(partition by d.dept_id  order by e.hire_date
                RANGE BETWEEN INTERVAL 1 YEAR PRECEDING AND CURRENT ROW) as employees_hired_last_1_year
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

Q043 – Above or Below Department Average ⭐⭐⭐ (Medium)

This is a very common interview question because it combines window functions + CASE expression.

Problem Statement

Using the employees table, return the following columns:

Employee Name,
Department Name,
Hire Date,
Salary,
Department Average Salary,
Salary Status

The Salary Status should contain:

'Above Average' → if employee salary is greater than the department average
'Below Average' → if employee salary is less than the department average
'Average' → if employee salary is exactly equal to the department average

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.hire_date, e.salary,
        cast(avg(e.salary) over(partition by d.dept_id) as decimal(10,2)) as dept_avg_salary,
            case 
                when e.salary > avg(e.salary) over(partition by d.dept_id) then 'Above Average'
                when e.salary < avg(e.salary) over(partition by d.dept_id) then 'Below Average'
                else 'Average'
            end as salary_status
        
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'


🚀 Q044 – Cumulative Percentage of Department Salary (Senior)

This is a favorite in finance, sales, and analytics interviews.

Problem:

For each employee, display:

Employee Name,
Department Name,
Salary,
Running Total Salary (ordered by salary descending),
Department Total Salary,
Cumulative Salary Percentage

The cumulative percentage should be calculated as:

(Running Total Salary / Department Total Salary) * 100

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        sum(e.salary) over(
                        partition by d.dept_id order by e.salary desc, e.emp_id
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                        ) as running_total_salary,

        sum(e.salary) over(partition by d.dept_id) as dept_total_salary,
        cast((running_total_salary / dept_total_salary)  * 100 as decimal(10,2)) as cumulative_salary_percentage
        
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

Q045 — FIRST_VALUE() and LAST_VALUE() ⭐⭐⭐⭐

This question catches many experienced SQL developers because of the behavior of LAST_VALUE().

Problem

Using the employees table, display:

Employee Name,
Department Name,
Salary,
Highest Salary in Department,
Lowest Salary in Department

But this time, do not use MAX() or MIN().

Use only:

FIRST_VALUE()
LAST_VALUE()

Rules:
✅ Use FIRST_VALUE()
✅ Use LAST_VALUE()
❌ Don't use MAX()
❌ Don't use MIN()
❌ No CTE
❌ No subquery

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        first_value(e.salary) over(partition by d.dept_id order by e.salary desc) as highest_salary,
        last_value(e.salary) over(
            partition by d.dept_id order by e.salary desc, e.emp_id
            rows between unbounded preceding and unbounded following
            ) as lowest_salary
        
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

Module 3.3 – LAG / LEAD

Topics:

Previous row
Next row
Difference from previous
Growth %
Month-over-month comparison

Already covered: Done

=====================================================================================================


Module 3.4 – FIRST_VALUE / LAST_VALUE

Topics:

Highest salary
Lowest salary
Latest transaction
Earliest transaction

Already covered: Done

Q045 — FIRST_VALUE() and LAST_VALUE() ⭐⭐⭐⭐

==================================================================================================


Module 3.5 – Distribution Functions

These are less common than LAG() or ROW_NUMBER(), but they do appear in senior interviews.

Q046 – CUME_DIST()

Q047 – PERCENT_RANK()

Q048 – NTILE()

Why this order?

Because they all answer a similar question:

"Where does this row stand relative to the others?"

CUME_DIST() → cumulative distribution
PERCENT_RANK() → percentile rank
NTILE() → bucket/group assignment

Learning them together makes the differences much easier to remember.

=======================================================================================================



Q046 — CUME_DIST() ⭐⭐⭐⭐ (Senior)
Business Scenario

The HR department wants to understand how employees compare to others within their own department based on salary.

Instead of assigning ranks, they want to know:

"What percentage of employees in this department have a salary less than or equal to this employee?"

This is exactly what CUME_DIST() calculates.

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        dense_rank() over(partition by d.dept_id order by e.salary) as rank,
        cume_dist() over(partition by d.dept_id order by e.salary) as cume_dist
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

Q047 — PERCENT_RANK() ⭐⭐⭐⭐ (Senior)
Business Scenario

The HR team wants to know:

"What percentage of employees does this employee outrank within their department?"

Unlike CUME_DIST(), which measures the cumulative proportion up to and including the current row, PERCENT_RANK() measures the employee's relative rank.

Problem Statement

Using the employees table, return:

Employee Name,
Department Name,
Salary,
DENSE_RANK(),
CUME_DIST(),
PERCENT_RANK()

Rules:
✅ Use DENSE_RANK()
✅ Use CUME_DIST()
✅ Use PERCENT_RANK()
✅ Partition by department
✅ Order by salary ASC (to match the previous business requirement)
❌ No CTE
❌ No subquery
❌ No GROUP BY

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        dense_rank() over(partition by d.dept_id order by e.salary) as rank,
        cume_dist() over(partition by d.dept_id order by e.salary) as cume_dist,
        percent_rank() over(partition by d.dept_id order by e.salary) as percent_rank
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'

Q048 — NTILE() ⭐⭐⭐⭐ (Senior)

This is the last distribution function.

Business Scenario

The HR department wants to categorize employees into salary quartiles.

Instead of knowing the exact rank, they want to group employees into:

Top 25%

Next 25%

Next 25%

Bottom 25%

These groups will be used for:

Bonus eligibility

Performance reviews

Leadership training

Salary benchmarking


Requirement Return:

Employee Name,
Department Name,
Salary,
Salary Quartile

Use: NTILE(4)

In [0]:
%sql

with salary_ranking as (
    select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        percent_rank() over(partition by d.dept_id order by e.salary desc) as rank
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'
)
select employee_name, department_name, salary, rank,
        case
            when rank <= .25 then 'Top 25%'
            when rank <= .50 then 'Top 50%'
            when rank <= .75 then 'Top 75%'
            else 'Top 100%'
        end as salary_quartile 
    from salary_ranking

In [0]:
%sql
select e.emp_name as employee_name, d.dept_name as department_name, e.salary,
        ntile(4) over(
                    partition by d.dept_id order by e.salary desc, emp_id asc
                    ) as salary_quartile
    from sql_interview.employees e
        inner join sql_interview.departments d
            on e.dept_id = d.dept_id
                and e.is_active = 'Y'